# Clinical Mortality and Length-of-Stay Modeling

This notebook runs the portfolio version of the project end to end: it starts from a local raw/harmonized admissions CSV, builds leakage-safe patient-level splits, trains XGBoost and a lightweight tabular Transformer, and compares mortality and length-of-stay (LOS) metrics.

## 1. Setup

Place the local admissions file at `data/raw/unified_patient_admissions_ccs.csv`. Patient-level data and generated outputs are ignored by git.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocess_split_first import SplitConfig, build_splits
from src.modeling import (
    load_model_matrices,
    train_tabular_transformer_models,
    train_xgboost_models,
)

RAW_CSV = PROJECT_ROOT / 'data/raw/unified_patient_admissions_ccs.csv'
OUTPUT_DIR = PROJECT_ROOT / 'data/processed'
REPORTS_DIR = PROJECT_ROOT / 'reports'
REPORTS_DIR.mkdir(exist_ok=True)

RAW_CSV

## 2. Build Leakage-Safe Splits

The split happens before imputation or encoding. All admissions from a single `subject_id` stay in one split, and the preprocessing object is fit only on the training split.

In [ ]:
if not RAW_CSV.exists():
    raise FileNotFoundError(
        f'Expected local input file at {RAW_CSV}. Copy your admissions CSV into data/raw first.'
    )

audit = build_splits(
    input_csv=RAW_CSV,
    output_dir=OUTPUT_DIR,
    config=SplitConfig(validation_size=0.15, test_size=0.15, random_state=42),
)
pd.DataFrame(audit['split_summary'])

## 3. Audit the Preprocessing

This audit is the core reproducibility artifact: it records split policy, excluded leakage-prone columns, selected feature counts, and high-cardinality columns deliberately left out of the tabular matrix.

In [ ]:
audit = json.loads((OUTPUT_DIR / 'preprocessing_audit.json').read_text())
print(json.dumps({
    'fit_policy': audit['fit_policy'],
    'excluded_from_features': audit['excluded_from_features'],
    'numeric_feature_count': audit['numeric_feature_count'],
    'categorical_feature_count': audit['categorical_feature_count'],
    'dropped_high_cardinality_categoricals': audit['dropped_high_cardinality_categoricals'],
}, indent=2))

## 4. Load Model Matrices

The generated matrices contain IDs, targets, and transformed model features. Modeling helpers remove IDs and targets from `X` automatically.

In [ ]:
bundle, frames = load_model_matrices(OUTPUT_DIR)
pd.DataFrame({
    split: {'rows': frame.shape[0], 'columns': frame.shape[1]}
    for split, frame in frames.items()
}).T

## 5. Train XGBoost

XGBoost is the interpretable structured-data baseline. Mortality uses class weighting and validation-selected F2 thresholding; LOS uses squared-error regression.

In [ ]:
xgb_mortality_model, xgb_los_model, xgb_metrics = train_xgboost_models(bundle)
xgb_metrics

## 6. Train Tabular Transformer

This compact Transformer treats each transformed tabular feature as a token. It is intentionally lightweight so the notebook remains runnable on a laptop. For a production multimodal Transformer, the sequence branch would consume diagnosis-token histories alongside the tabular branch.

In [ ]:
transformer_mortality_model, transformer_los_model, transformer_metrics = train_tabular_transformer_models(
    bundle,
    epochs=8,
    batch_size=256,
)
transformer_metrics

## 7. Compare Models

Mortality is evaluated with AUROC, AUPRC, F1, precision, recall, and the validation-selected threshold. LOS is evaluated with RMSE, MAE, and R2.

In [ ]:
comparison = pd.concat([xgb_metrics, transformer_metrics], ignore_index=True)
comparison.to_csv(REPORTS_DIR / 'model_comparison_metrics.csv', index=False)
comparison

## 8. Patient Leakage Check

The raw split files are retained locally so the split can be audited after modeling.

In [ ]:
subject_sets = {
    split: set(pd.read_csv(OUTPUT_DIR / f'{split}_raw_split.csv')['subject_id'])
    for split in ['train', 'val', 'test']
}

assert subject_sets['train'].isdisjoint(subject_sets['val'])
assert subject_sets['train'].isdisjoint(subject_sets['test'])
assert subject_sets['val'].isdisjoint(subject_sets['test'])

'No subject leakage across train/validation/test splits.'